# Delimitación automática de ambientes — Google Earth Engine + machine learning

Flujo por etapas. Cada etapa se puede volver a ejecutar sin repetir las anteriores
(por ejemplo, volver a clasificar con otro número de ambientes).

**Antes de empezar**
1. En el panel izquierdo de Colab abrí **Secrets (🔑)** y creá `GITHUB_TOKEN` con un token de GitHub de
   solo lectura para el repo. Activá el acceso del notebook a ese secret.
2. Prepará las entradas:
   - **Perímetro del lote**: `.zip` con el SHP (`.shp`, `.shx`, `.dbf`, `.prj`) o un `.geojson`.
   - **Excel** con la hoja `imagenes` (fecha, cultivo, campaña) y, si hay, la hoja `rendimiento`
     (archivo, campaña, cultivo, columna). Podés bajar la plantilla en el paso 2.
   - **Mapas de rendimiento** limpios (opcional): un `.zip` por SHP de puntos. El nombre del `.shp` tiene que
     coincidir con la columna `archivo` del Excel.

In [ ]:
#@title 1. Instalación (clona el repo y las dependencias)
RAMA = "claude/agricultural-environment-delimitation-j7g2x6"  #@param {type:"string"}
REPO = "pablopaparelli-tech/Ambientado-autom-tico-rep"
DESTINO = "/content/ambientado"

import os, sys, subprocess
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
url = f"https://{token}@github.com/{REPO}.git"
if not os.path.exists(DESTINO):
    r = subprocess.run(["git", "clone", "-q", "-b", RAMA, url, DESTINO], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "-C", DESTINO, "pull", "-q", url, RAMA], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError("No se pudo clonar/actualizar el repo: " + r.stderr.replace(token, "***"))
subprocess.run(["git", "-C", DESTINO, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"])
del token, url

%pip install -q -r {DESTINO}/requirements.txt
sys.path.insert(0, DESTINO)
print("Listo.")

In [ ]:
#@title 2. (Opcional) Descargar la plantilla de Excel
from google.colab import files
files.download(f"{DESTINO}/plantillas/entrada_fechas_rendimiento.xlsx")

In [ ]:
#@title 3a. Subir perímetro del lote (.zip con SHP o .geojson) y Excel (.xlsx)
import zipfile, shutil, glob
from google.colab import files

ENTRADAS = "/content/entradas"
REND_DIR = f"{ENTRADAS}/rendimiento"
os.makedirs(REND_DIR, exist_ok=True)
LOTE_PATH = EXCEL_PATH = None
for nombre in files.upload():
    ruta = os.path.join(ENTRADAS, nombre)
    shutil.move(nombre, ruta)
    if nombre.lower().endswith((".xlsx", ".xls")):
        EXCEL_PATH = ruta
    else:
        LOTE_PATH = ruta
print("Lote :", LOTE_PATH)
print("Excel:", EXCEL_PATH)

In [ ]:
#@title 3b. (Opcional) Subir mapas de rendimiento (.zip con SHP de puntos, uno o varios)
for nombre in files.upload():
    with zipfile.ZipFile(nombre) as z:
        z.extractall(REND_DIR)
    os.remove(nombre)
# los SHP pueden venir dentro de subcarpetas del zip
for shp in glob.glob(f"{REND_DIR}/**/*.shp", recursive=True):
    for ext in ("shp", "shx", "dbf", "prj", "cpg"):
        src = shp[:-3] + ext
        if os.path.exists(src) and os.path.dirname(src) != REND_DIR:
            shutil.move(src, REND_DIR)
print("Rendimiento:", sorted(os.path.basename(p) for p in glob.glob(f"{REND_DIR}/*.shp")))

In [ ]:
#@title 4. Configuración
from ambientacion import Config, Proyecto
from IPython.display import display

establecimiento = "San Remigio"  #@param {type:"string"}
lote = "Lote 1"  #@param {type:"string"}
analista = ""  #@param {type:"string"}
gee_project = "ee-pablopaparelli"  #@param {type:"string"}
#@markdown **Sentinel-2**
ventana_dias = 10  #@param {type:"integer"}
buffer_nubes_m = 20  #@param {type:"number"}
#@markdown **Topografía y anegamiento**
tpi_radio_m = 150  #@param {type:"number"}
usar_l9 = True  #@param {type:"boolean"}
#@markdown **Clustering y polígonos**
k_min = 2  #@param {type:"integer"}
k_max = 6  #@param {type:"integer"}
area_min_ambiente_pct = 5  #@param {type:"number"}
umm_ha = 0.5  #@param {type:"number"}

cfg = Config(
    gee_project=gee_project, establecimiento=establecimiento, lote=lote, analista=analista,
    lote_path=LOTE_PATH, excel_path=EXCEL_PATH, rendimiento_dir=REND_DIR, salida_dir="/content/salidas",
    ventana_dias=ventana_dias, buffer_nubes_m=buffer_nubes_m, tpi_radio_m=tpi_radio_m, usar_l9=usar_l9,
    k_min=k_min, k_max=k_max, area_min_ambiente_pct=area_min_ambiente_pct, umm_ha=umm_ha,
)
p = Proyecto(cfg)
p.cargar_entradas()
display(p.tabla_img_entrada)
if p.tabla_rend is not None:
    display(p.tabla_rend)

In [ ]:
#@title 5. Earth Engine
p.inicializar_ee()

## Capas de información

In [ ]:
#@title 6. Sentinel-2: fechas sin nubes/sombras e índices (NDVI, NDMI, CIre)
tabla = p.capas_sentinel()
display(tabla)

In [ ]:
#@title 7. Topografía: DEM Copernicus GLO-30 y TPI
p.capas_topografia()

In [ ]:
#@title 8. Frecuencia de anegamiento (Landsat 5/8/9)
p.capa_anegamiento()

In [ ]:
#@title 9. Rendimiento (si se cargó en el Excel)
if p.tabla_rend is not None:
    p.capas_rendimiento()
    display(p.tabla_rend)
else:
    print("Sin mapas de rendimiento.")

## Análisis exploratorio

In [ ]:
#@title 10. Normalización y EDA
import matplotlib.pyplot as plt
disponibles = p.normalizar()
res = p.eda()
display(res["resumen"])
for f in res["figs"].values():
    display(f); plt.close(f)
print("VIF (valores > 10 indican redundancia):")
display(res["vif"].to_frame())
print("Pares con |correlación| ≥ 0,85:")
for a, b, r in res["pares"]:
    print(f"  {a} ~ {b}: {r}")

## Clustering

En el paso 11 elegí las variables que entran al clustering (por defecto, todas). Para sacar variables
redundantes del EDA, escribí la lista separada por comas, por ejemplo: `ndvi_rel, ndvi_cv, cire_cv, tpi, anegamiento`.

In [ ]:
#@title 11. Variables y número de ambientes
variables = ""  #@param {type:"string"}
claves = [v.strip() for v in variables.split(",") if v.strip()] or disponibles
k_df, votos, k_sug = p.evaluar_k(claves)
display(p.fig_k)
display(k_df.round(3))
print("Votos:", votos, "→ k sugerido:", k_sug)

In [ ]:
#@title 12. Clasificación, ranking, nombres y variables explicativas
k_elegido = 0  #@param {type:"integer"}
#@markdown 0 = usar el k sugerido
nombres = p.clasificar(k_elegido or None)
print("Ambientes (1 = mayor potencial):", nombres, "· criterio:", p.criterio)
exp = p.explicar()
display(exp["fig"])
display(exp["tabla"])
print(f"Exactitud del Random Forest: {exp['exactitud_rf']:.1%}")

In [ ]:
#@title 13. Polígonos generalizados y validación topológica
from ambientacion import mapas
gdf, validacion = p.vectorizar()
display(mapas.mapa_ambientes(gdf, p.lote, f"Ambientes — {cfg.lote}"))
display(gdf.drop(columns="geometry").round(2))
display(validacion)
from ambientacion.informe import tabla_estadisticas_informe
display(tabla_estadisticas_informe(p))
for t in p.textos.values():
    print("•", t)

In [ ]:
#@title 14. Visor interactivo
normalizadas = False  #@param {type:"boolean"}
p.visor(normalizadas)

## ¿Generar el informe?

Revisá el mapa, las estadísticas y el visor.
- **Si hay que reevaluar:** volvé al paso 11 (variables) o al 12 (otro k) y ejecutá de nuevo desde ahí.
- **Si está bien:** activá `generar_informe` y ejecutá el paso 15.

In [ ]:
#@title 15. Informe (HTML + PDF) y productos
generar_informe = False  #@param {type:"boolean"}
if generar_informe:
    out = p.informe()
    from google.colab import files
    files.download(out["zip"])
else:
    print("Informe no generado. Activá 'generar_informe' cuando el análisis esté aprobado.")